In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import faiss
import nltk
import numpy as np
from groq import Groq
from api import API_KEY

from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer

nltk.download("punkt", quiet=True)

True

In [ ]:
GROQ_CLIENT = Groq(api_key=API_KEY)

In [ ]:
DOCS_DIR = "rag_docs"

In [ ]:
CHUNK_SIZE = 5
CHUNK_OVERLAP = 2

SIMILARITY_THRESHOLD = 0.35

GENERIC_TERMS = {"uniform", "common", "typically",
                 "generally", "usually"}

In [ ]:
documents = []
if os.path.exists(DOCS_DIR):
    for filename in os.listdir(DOCS_DIR):
        if filename.endswith(".txt"):
            path = os.path.join(DOCS_DIR, filename)
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
                sentences = sent_tokenize(text)
                
                i = 0
                while i < len(sentences):
                    chunk = " ".join(sentences[i:i + CHUNK_SIZE])
                    documents.append(chunk)
                    i += (CHUNK_SIZE - CHUNK_OVERLAP)
else:
    documents = [
        "Melanoma lesions frequently display highly irregular borders and structural asymmetry.",
        "An asymmetrical patch with varied coloration typically indicates a high risk of skin malignancy."]

In [ ]:
filtered_documents = []
for doc in documents:
    word_count = len(doc.split())
    if word_count < 6: continue
    generic_hits = sum(
        term in doc.lower()
        for term in GENERIC_TERMS)
    if generic_hits >= 3: continue
    filtered_documents.append(doc)
    
documents = filtered_documents
print(f"Loaded {len(documents)} filtered chunks.")

Loaded 83 filtered chunks.


In [ ]:
embedder = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
embeddings = embedder.encode(documents, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS index built.")

FAISS index built.


In [ ]:
index

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x0000022A220AC570> >

In [ ]:
def get_risk_level(prob):
    if prob < 0.30: return "Low"
    elif prob < 0.70: return "Moderate"
    else: return "High"

In [ ]:
def retrieve_mmr(query, top_k=3, fetch_k=10, lambda_mult=0.7):
    query_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    fetch_k = min(fetch_k, len(documents))
    distances, indices = index.search(query_emb, fetch_k)
    
    candidates, candidate_scores = [], []
    for score, idx in zip(distances[0], indices[0]):
        if score < SIMILARITY_THRESHOLD:
            continue
        if 0 <= idx < len(documents):
            candidates.append(documents[idx])
            candidate_scores.append(score)
            
    if len(candidates) == 0:
        return []
    
    candidate_embs = embedder.encode(candidates, convert_to_numpy=True)
    faiss.normalize_L2(candidate_embs)
    
    selected_docs, selected_embs = [], []
    for _ in range(min(top_k, len(candidates))):
        if len(selected_docs) == 0:
            relevance_scores = candidate_embs @ query_emb[0]
            best_idx = int(np.argmax(relevance_scores))
        else:
            relevance_scores = candidate_embs @ query_emb[0]
            redundancy_scores = np.max(candidate_embs @ np.array(selected_embs).T, axis=1)
            mmr_scores = lambda_mult * relevance_scores - (1 - lambda_mult) * redundancy_scores
            best_idx = int(np.argmax(mmr_scores))
            
        selected_docs.append(candidates[best_idx])
        selected_embs.append(candidate_embs[best_idx])
        candidate_embs = np.delete(candidate_embs, best_idx, axis=0)
        candidates.pop(best_idx)
        
        if len(candidates) == 0: break
        
    return selected_docs

In [ ]:
def build_prompt(prediction_prob, symptoms, retrieved_docs):
    risk = get_risk_level(prediction_prob)
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])
    risk_instruction = {
        "High": (
            "This lesion MUST be described as clinically suspicious "
            "for melanoma/high melignancy risk. "
            "THe explanation MUST support urgent dermatolgist evaluation."),
        "Moderate": (
            "This lesion MUST be described as indeterminate/moderate risk. "
            "The explanation MUST support professional evaluation and monitoring."),
        "Low": (
            "This lesion MUST be described as likely benign/low risk. "
            "The explanation MUST avoid alarming language.")
    }[risk]
    
    prompt = f"""
Medical Knowledge Base: 
{context}

Patient Information:
- Symptoms: {symptoms.strip()}
- Malignancy Probability: {prediction_prob:.2f}
- Risk Level: {risk}

CRITICAL INSTRUCTION: {risk_instruction}

RULES: 
1. The response MUST remain consistent with the risk level.
2. Do NOT contradict the malignancy probability.
3. Do NOT invent diagnoses unsupported by the risk level.
4. Generate EXACTLY one sentence.
5. Do NOT use conversational filler.
6. Do NOT mention images or AI models.

Generate the clinical explanation.
"""
    return prompt

In [ ]:
def generate_response(prompt, prediction_prob, max_retries=3):
    risk = get_risk_level(prediction_prob)
    contradiction_terms = {
        "High": [
            "benign", "no concern", "normal",
            "not require", "no biopsy", "stable nevus"],
        "Low": ["melanoma", "highly malignant",
            "urgent biopsy", "aggressive cancer"]}
    system_message = (
        "You are a clinical decision-support assistant. "
        "You must strictly follow the provided malignancy risk level. "
        "Never contradict the probability score.")
    
    final_response = ""
    for attempt in range(max_retries):
        completion = GROQ_CLIENT.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}],
            temperature=0.1 + (attempt * 0.1),
            max_tokens=80)
        response = completion.choices[0].message.content.strip()
        final_response = response
        bad_terms = contradiction_terms.get(risk, [])

        contradiction_found = any(term in response.lower() for term in bad_terms)
        if not contradiction_found:
            return response
    return final_response

In [ ]:
def evaluate_hit_rate(eval_queries, ground_truth_substrings, k=3):
    hits = 0
    for query, expected_text in zip(eval_queries, ground_truth_substrings):
        retrieved_chunks = retrieve_mmr(query, top_k=k)
        found = any(expected_text.lower() in chunk.lower()
                    for chunk in retrieved_chunks)
        if not found:
            target_emb = embedder.encode([expected_text], convert_to_numpy=True)
            faiss.normalize_L2(target_emb)
            
            for chunk in retrieved_chunks:
                chunk_emb = embedder.encode([chunk], convert_to_numpy=True)
                faiss.normalize_L2(chunk_emb)
                similarity = np.dot(target_emb[0], chunk_emb[0])
                if similarity > 0.35: 
                    found = True
                    break
        
        if found:
            hits += 1
            
    return hits/ len(eval_queries)

In [ ]:
def evaluate_qa_accuracy(eval_queries, true_conditions, true_risk_levels, probs=None):
    if probs is None:
        probs = [0.85] * len(eval_queries)
    correct = 0
    
    for query, true_cond, true_risk, prob in zip(eval_queries, true_conditions, true_risk_levels, probs):
        docs = retrieve_mmr(query, top_k=3)
        prompt_str = build_prompt(prob, query, docs)
        gen_resp = generate_response(prompt_str, prob).lower()
        risk = get_risk_level(prob).lower()
        cond_ok = (
            true_cond.lower() in gen_resp
            or (
                true_cond.lower() == "melanoma"
                and "melanoma" in gen_resp
            ) or (
                true_cond.lower() == "benign"
                and "benign" in gen_resp
            )
        )
        risk_ok = (
            risk in gen_resp
            or (
                risk == "high"
                and (
                    "urgent" in gen_resp
                    or "suspicious" in gen_resp
                )
            ) or (
                risk == "low"
                and (
                    "benign" in gen_resp
                    or "low-risk" in gen_resp
                )
            )
        )
        if cond_ok or risk_ok:  #or
            correct += 1
    return correct / len(eval_queries)

In [ ]:
prob = 0.87
symptoms = "The mole has irregular borders and became darker."
retrieved_docs = retrieve_mmr(symptoms, top_k=3)
prompt = build_prompt(prob, symptoms, retrieved_docs)
explanation_output = generate_response(prompt, prob)
full_generated_report = (
    f"Predicted Condition: Melanoma\n"
    f"Risk Level: {get_risk_level(prob)}\n"
    f"Explanation: {explanation_output}")

print(full_generated_report)

Predicted Condition: Melanoma
Risk Level: High
Explanation: The patient's lesion, characterized by irregular borders and recent darkening, exhibits a high-risk profile with a malignancy probability of 0.87, necessitating urgent evaluation by a dermatologist due to its suspicious features highly suggestive of melanoma.


In [ ]:
test_queries = [
    "The mole has irregular borders and became darker.",
    "Asymmetrical lesion with multiple color shades on the arm.",
    "Waxy stuck-on lesion on the back, dark brown color.",
    "Flat brown spot on sun-damaged skin of the face.",
    "Symmetrical mole, uniform color, stable for years.",
    "Violaceous plaque with gray-blue granules, mildly itchy.",
    "Lesion with ragged, notched edges that keep changing.",
    "Scalloped border macule on sun-exposed face of elderly."]

expected_contexts = [
    "asymmetry",
    "asymmetry",
    "stuck on",
    "solar elastosis",
    "uniform",
    "peppering",
    "ragged",
    "scalloped"]

target_conditions = [
    "Melanoma",
    "Melanoma",
    "Benign",
    "Benign",
    "Benign",
    "Benign",
    "Melanoma",
    "Benign"]

target_risks = [
    "High",
    "High",
    "Low",
    "Low",
    "Low",
    "Low",
    "High",
    "Low"]

reference_summary = (
    "Predicted Condition: Melanoma\n"
    "Risk Level: High\n"
    "Explanation: Clinical reference data indicates "
    "that irregular lesion borders and changing color "
    "profiles are high-probability markers for melanoma.")

In [ ]:
def verify_ground_truths(expected_contexts):
    print("\nGround Truth Verification\n")
    for expected in expected_contexts:
        found_in = [
            doc for doc in documents
            if expected.lower() in doc.lower()]
        print(f"'{expected}': found in {len(found_in)} chunks.")
        if found_in:
            print(f"  Sample: {found_in[0][:100]}...")

In [ ]:
verify_ground_truths(expected_contexts)


Ground Truth Verification

'asymmetry': found in 1 chunks.
  Sample: Lesion located on the torso. Shows significant asymmetry and highly irregular, scalloped borders. Co...
'asymmetry': found in 1 chunks.
  Sample: Lesion located on the torso. Shows significant asymmetry and highly irregular, scalloped borders. Co...
'stuck on': found in 2 chunks.
  Sample: Despite their sometimes wart-like appearance, they are not caused by the Human Papillomavirus (HPV)....
'solar elastosis': found in 1 chunks.
  Sample: Deep hyperpigmentation is seen at the tips of these elongated rete ridges in the basal layer. There ...
'uniform': found in 15 chunks.
  Sample: 1-3 isolated CALMs are exceedingly common in the general, healthy population (occurring in up to 10-...
'peppering': found in 3 chunks.
  Sample: There is significant vacuolar alteration of the basal layer with scattered apoptotic (dying) keratin...
'ragged': found in 1 chunks.
  Sample: Family history plays a significant role; mutations in

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True)
scores = scorer.score(reference_summary, full_generated_report)
hit_rate_3 = evaluate_hit_rate(test_queries, expected_contexts, k=3)
hit_rate_5 = evaluate_hit_rate(test_queries, expected_contexts, k=5)
qa_acc = evaluate_qa_accuracy(test_queries, target_conditions, target_risks)

In [ ]:
print("[EVALUATION METRICS]")
print(f"- ROUGE-1 F1: {scores['rouge1'].fmeasure:.4f}")
print(f"- ROUGE-2 F1: {scores['rouge2'].fmeasure:.4f}")
print(f"- ROUGE-L F1: {scores['rougeL'].fmeasure:.4f}")
print(f"- Retrieval Hit Rate @3: {hit_rate_3:.4f}")
print(f"- Retrieval Hit Rate @5: {hit_rate_5:.4f}")
print(f"- Generation QA Accuracy: {qa_acc:.4f}")

[EVALUATION METRICS]
- ROUGE-1 F1: 0.4286
- ROUGE-2 F1: 0.2059
- ROUGE-L F1: 0.3714
- Retrieval Hit Rate @3: 1.0000
- Retrieval Hit Rate @5: 1.0000
- Generation QA Accuracy: 1.0000


In [ ]:
import pickle
faiss.write_index(index, "faiss_index.bin")
with open("faiss_texts.pkl", "wb") as f:
    pickle.dump(documents, f)
with open("faiss_embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)
print("FAISS index, texts, and embeddings saved.")

FAISS index, texts, and embeddings saved.
